In [79]:
import pandas as pd
import numpy as np
import random
import json
import time
from datetime import datetime, timedelta
from collections import defaultdict, Counter

In [ ]:

!pip install -q kaggle kagglehub python-dotenv

import os
from dotenv import load_dotenv

# Load .env variables
load_dotenv()

# Set Kaggle credentials
os.environ['KAGGLE_USERNAME'] = os.getenv("KAGGLE_USERNAME")
os.environ['KAGGLE_KEY'] = os.getenv("KAGGLE_KEY")


import kagglehub

goemotions_path1 = kagglehub.dataset_download('bhadramohit/social-media-usage-datasetapplications')
goemotions_path2 = kagglehub.dataset_download('khushikyad001/screen-time-and-app-usage-dataset-iosandroid')
goemotions_path3 = kagglehub.dataset_download('anandshaw2001/mobile-apps-screentime-analysis')
goemotions_path4 = kagglehub.dataset_download('dem0nking/mobile-games-android-and-ios-rating-dataset')



Using Colab cache for faster access to the 'social-media-usage-datasetapplications' dataset.
Using Colab cache for faster access to the 'screen-time-and-app-usage-dataset-iosandroid' dataset.
Using Colab cache for faster access to the 'mobile-apps-screentime-analysis' dataset.
Using Colab cache for faster access to the 'mobile-games-android-and-ios-rating-dataset' dataset.


In [81]:
print(goemotions_path3)

/kaggle/input/mobile-apps-screentime-analysis


In [82]:
dataset1_path = '/kaggle/input/mobile-games-android-and-ios-rating-dataset'
!ls "{dataset1_path}"

ratings.csv


In [83]:
DATASET_1 = '/kaggle/input/social-media-usage-datasetapplications/social_media_usage.csv'
DATASET_2 = '/kaggle/input/screen-time-and-app-usage-dataset-iosandroid/screen_time_app_usage_dataset.csv'
DATASET_3 = '/kaggle/input/mobile-apps-screentime-analysis/screentime_analysis.csv'
DATASET_4 = '/kaggle/input/mobile-games-android-and-ios-rating-dataset/ratings.csv'

OUTPUT_FILE = "synthetic_usage_final.json"


In [84]:
def preprocess_dataset1(path):

    df = pd.read_csv(path)

    df = df.rename(columns={
        "App": "app_name",
        "Daily_Minutes_Spent": "screen_time_min"
    })

    df["screen_time_min"] = pd.to_numeric(
        df["screen_time_min"], errors="coerce"
    )

    df["date"] = None
    df["category"] = "Social"
    df["source"] = "dataset1"


    return df[["date", "app_name", "screen_time_min", "category"]]


In [85]:
def preprocess_dataset2(path):

    df = pd.read_csv(path)

    df["screen_time_min"] = pd.to_numeric(
        df["screen_time_min"], errors="coerce"
    )

    df["date"] = None
    df["source"] = "dataset2"

    return df[["date", "app_name", "screen_time_min", "category"]]


In [86]:
def preprocess_dataset3(path):

    df = pd.read_csv(path)

    df = df.rename(columns={
        "App": "app_name",
        "Usage (minutes)": "screen_time_min",
        "Date": "date"
    })

    df["screen_time_min"] = pd.to_numeric(
        df["screen_time_min"], errors="coerce"
    )

    df["category"] = "Unknown"
    df["source"] = "dataset3"


    return df[["date", "app_name", "screen_time_min", "category"]]


In [87]:
def preprocess_dataset4(path):

    df = pd.read_csv(path)

    df = df.rename(columns={
        "Game Name": "app_name",
        "Genre": "category"
    })

    df["screen_time_min"] = np.nan
    df["date"] = None

    # mark as dataset 4
    df["source"] = "dataset4"

    return df[["date", "app_name", "screen_time_min", "category", "source"]]

In [88]:
def load_all_data():

    d1 = preprocess_dataset1(DATASET_1)
    d2 = preprocess_dataset2(DATASET_2)
    d3 = preprocess_dataset3(DATASET_3)
    d4 = preprocess_dataset4(DATASET_4)

    df = pd.concat([d1, d2, d3, d4], ignore_index=True)

    df = df.dropna(subset=["app_name"])

    return df

In [89]:
def normalize_app_name(name):
    return (
        str(name)
        .strip()
        .lower()
        .replace("_", " ")
    )

In [90]:
KNOWN_APP_CATEGORIES = {
    "instagram": "Social",
    "facebook": "Social",
    "whatsapp": "Social",
    "snapchat": "Social",
    "twitter": "Social",
    "tiktok": "Social",

    "youtube": "Entertainment",
    "netflix": "Entertainment",
    "spotify": "Entertainment",
    "prime video": "Entertainment",

    "gmail": "Productivity",
    "outlook": "Productivity",
    "slack": "Productivity",
    "zoom": "Productivity",
    "notion": "Productivity",

    "subway surfers": "Gaming",
    "pubg": "Gaming",
    "clash of clans": "Gaming",
    "candy crush": "Gaming"
}

In [91]:
CATEGORY_KEYWORDS = {
    "Social": [
        "chat", "message", "social", "facebook",
        "instagram", "snap", "whatsapp", "messenger"
    ],
    "Gaming": [
        "game", "clash", "battle", "war",
        "racing", "puzzle"
    ],
    "Productivity": [
        "mail", "office", "docs", "drive",
        "calendar", "meet", "zoom", "slack"
    ],
    "Entertainment": [
        "video", "music", "stream", "tv",
        "netflix", "spotify"
    ]
}

In [92]:
def learn_patterns(df):

    app_usage = defaultdict(list)
    app_presence = Counter()
    category_votes = defaultdict(list)
    dataset4_apps = set()

    for _, row in df.iterrows():

        raw_name = row["app_name"]
        app = normalize_app_name(raw_name)

        # detect dataset 4
        if row.get("source") == "dataset4":
            dataset4_apps.add(app)

        if pd.notna(row["screen_time_min"]):
            ms = row["screen_time_min"] * 60000
            app_usage[app].append(ms)

        if pd.notna(row["category"]):
            category_votes[app].append(row["category"])

        app_presence[app] += 1

    # Resolve categories properly
    app_category = {}

    for app in app_presence:

        # FORCE dataset4 apps to Gaming
        if app in dataset4_apps:
            app_category[app] = "Gaming"
            continue

        # Known mapping wins
        if app in KNOWN_APP_CATEGORIES:
            app_category[app] = KNOWN_APP_CATEGORIES[app]
            continue

        # Majority vote from datasets
        if category_votes[app]:
            most_common = Counter(category_votes[app]).most_common(1)[0][0]
            if most_common != "Unknown":
                app_category[app] = most_common
                continue

        # Keyword inference
        assigned = False
        for category, keywords in CATEGORY_KEYWORDS.items():
            if any(k in app for k in keywords):
                app_category[app] = category
                assigned = True
                break

        # Final fallback
        if not assigned:
            app_category[app] = "Unknown"

    total = sum(app_presence.values())

    app_prob = {
        app: c / total
        for app, c in app_presence.items()
    }

    return {
        "app_usage": app_usage,
        "app_category": app_category,
        "app_prob": app_prob
    }

In [93]:
WEEKDAY_CATEGORY_BIAS = {
    "Productivity": 1.3,
    "Utilities": 1.2,
    "Social": 1.0,
    "Entertainment": 0.9,
    "Gaming": 0.9
}

WEEKEND_CATEGORY_BIAS = {
    "Productivity": 0.8,
    "Utilities": 0.9,
    "Social": 1.1,
    "Entertainment": 1.3,
    "Gaming": 1.4
}

In [94]:
def choose_app_with_context(patterns, is_weekend, chosen):

    category_apps = patterns["category_apps"]
    app_category = patterns["app_category"]
    app_prob = patterns["app_prob"]

    # Pick bias dictionary
    bias = WEEKEND_CATEGORY_BIAS if is_weekend else WEEKDAY_CATEGORY_BIAS

    # Compute category probabilities dynamically
    category_weights = {}

    for cat, apps in category_apps.items():
        base = sum(app_prob.get(a,0) for a in apps)
        category_weights[cat] = base * bias.get(cat,1.0)

    cats = list(category_weights.keys())
    weights = np.array(list(category_weights.values()))
    weights = weights / weights.sum()

    chosen_cat = np.random.choice(cats, p=weights)

    # apps inside category
    apps_in_cat = list(set(category_apps[chosen_cat]))

    # Slight co-occurrence boost:
    # If already selected app from this category → boost probability
    if any(app_category.get(a)==chosen_cat for a in chosen):
        weights = [app_prob[a]*1.4 for a in apps_in_cat]
    else:
        weights = [app_prob[a] for a in apps_in_cat]

    weights = np.array(weights)
    weights = weights / weights.sum()

    return np.random.choice(apps_in_cat, p=weights)

In [95]:
TIME_BUCKETS = {
    "morning": (6, 11),
    "afternoon": (11, 17),
    "evening": (17, 22),
    "late_night": (22, 2)
}

In [96]:
CATEGORY_TIME_WEIGHTS = {
    "Productivity": {
        "morning": 0.4,
        "afternoon": 0.4,
        "evening": 0.15,
        "late_night": 0.05
    },
    "Social": {
        "morning": 0.15,
        "afternoon": 0.3,
        "evening": 0.4,
        "late_night": 0.15
    },
    "Gaming": {
        "morning": 0.05,
        "afternoon": 0.2,
        "evening": 0.5,
        "late_night": 0.25
    },
    "Entertainment": {
        "morning": 0.05,
        "afternoon": 0.25,
        "evening": 0.5,
        "late_night": 0.2
    },
    "Unknown": {
        "morning": 0.25,
        "afternoon": 0.25,
        "evening": 0.25,
        "late_night": 0.25
    }
}

In [97]:
def sample_time_of_day(category):

    weights = CATEGORY_TIME_WEIGHTS.get(
        category,
        CATEGORY_TIME_WEIGHTS["Unknown"]
    )

    buckets = list(weights.keys())
    probs = np.array(list(weights.values()))
    probs /= probs.sum()

    chosen_bucket = np.random.choice(buckets, p=probs)

    start_hour, end_hour = TIME_BUCKETS[chosen_bucket]

    if chosen_bucket == "late_night":
        # wrap around midnight
        hour = random.choice(list(range(22, 24)) + list(range(0, 2)))
    else:
        hour = random.randint(start_hour, end_hour - 1)

    minute = random.randint(0, 59)
    second = random.randint(0, 59)

    return hour, minute, second

In [98]:
def sample_usage(samples):

    if len(samples) < 3:
        return int(random.randint(1, 180) * 60000)

    logs = np.log(np.array(samples) + 1)
    mu, sigma = logs.mean(), logs.std()

    val = np.random.lognormal(mu, sigma)

    return int(max(60000, val))


In [99]:
def choose_app(app_prob):

    apps = list(app_prob.keys())
    probs = np.array(list(app_prob.values()))
    probs /= probs.sum()

    return np.random.choice(apps, p=probs)


In [100]:
def synthesize_day(date_str, patterns, min_apps=5, max_apps=20):
    date_obj = datetime.strptime(date_str, "%Y-%m-%d")
    weekday = date_obj.weekday()  # 0=Mon, 6=Sun
    is_weekend = weekday >= 5
    is_friday = weekday == 4

    n_apps = random.randint(min_apps, max_apps)  # use dynamic min/max

    chosen = set()
    apps = []

    while len(chosen) < n_apps:
        app = choose_app_with_context(patterns, is_weekend, chosen)

        if app in chosen:
            continue

        chosen.add(app)
        category = patterns["app_category"].get(app, "Unknown")

        # Optional: keep your weekday/weekend logic
        if is_weekend:
            if category in ["Productivity"] and random.random() < 0.5:
                continue
        else:
            if category in ["Gaming"] and random.random() < 0.2:
                continue

        # Friday entertainment boost
        if is_friday and category in ["Entertainment", "Gaming"]:
            if random.random() < 0.3:
                n_apps += 1

        samples = patterns["app_usage"].get(app, [])
        time_ms = sample_usage(samples)

        hour, minute, second = sample_time_of_day(category)
        usage_time = date_obj.replace(hour=hour, minute=minute, second=second)
        timestamp = int(usage_time.timestamp() * 1000)

        pkg = "com.synthetic." + app.lower().replace(" ", "")

        apps.append({
            "package": pkg,
            "time_ms": time_ms,
            "time_min": time_ms // 60000,
            "last_used": timestamp,
            "app_name": app,
            "category": category
        })

    # Scale app times randomly to vary total_time_min
    scale_factors = [random.uniform(0.8, 1.2) for _ in apps]
    total_ms = 0
    for i, sf in enumerate(scale_factors):
        new_time = int(apps[i]["time_ms"] * sf)
        apps[i]["time_ms"] = new_time
        apps[i]["time_min"] = new_time // 60000
        total_ms += new_time

    apps.sort(key=lambda x: x["last_used"])

    return {
        "date": date_str,
        "weekday": weekday,
        "is_weekend": is_weekend,
        "total_time_min": total_ms // 60000,
        "app_count": len(apps),
        "apps": apps
    }

In [101]:
def synthesize_range(start_date, n_days, patterns, min_apps=5, max_apps=20):
    """
    Generate synthetic data for a range of dates.
    """
    start = datetime.strptime(start_date, "%Y-%m-%d")
    output = []

    for i in range(n_days):
        d = (start + timedelta(days=i)).strftime("%Y-%m-%d")
        day_data = synthesize_day(d, patterns, min_apps=min_apps, max_apps=max_apps)
        output.append(day_data)

    return output


In [102]:
def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    elif isinstance(obj, set):
        return list(obj)
    elif isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    else:
        return obj


In [103]:
def main():

    df = load_all_data()

    patterns = learn_patterns(df)

    synthetic = synthesize_range(
        "2026-01-01",
        365,
        patterns
    )

    with open(OUTPUT_FILE, "w") as f:
        json.dump(synthetic, f, indent=2)

    print("Synthetic data generated.")



In [104]:
if __name__ == "__main__":
    main()


KeyError: 'category_apps'

In [ ]:
# Read the JSON
with open('/content/synthetic_usage_final.json', 'r') as f:
    data = json.load(f)

# Normalize the nested structure
df = pd.json_normalize(
    data,
    record_path='apps',  # Path to the nested array
    meta=['date', 'weekday', 'is_weekend', 'total_time_min', 'app_count']  # Metadata to include
)

print(df.head())
print(f"\nShape: {df.shape}")

In [ ]:
df[df['date']=='2026-12-31']